In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src.config.settings import (
    DATABASE_NAME,
    SILVER_TABLE,
    GOLD_DAILY_METRICS_TABLE,
    GOLD_HOURLY_METRICS_TABLE
)

In [0]:
spark = SparkSession.builder.appName("ifood-case-gold").getOrCreate()

In [0]:
def validate_silver_table_exists() -> None:
    """
    Valida se a tabela Silver existe antes de gerar a camada Gold.
    """

    tables = [
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {DATABASE_NAME}").collect()
    ]

    silver_table_name = SILVER_TABLE.split(".")[-1]

    if silver_table_name not in tables:
        raise ValueError(
            f"A tabela Silver {SILVER_TABLE} não existe. "
            "Execute primeiro o pipeline principal."
        )

    print(f"Tabela Silver localizada com sucesso: {SILVER_TABLE}")


def drop_gold_tables_if_exists() -> None:
    """
    Remove as tabelas Gold para garantir reprocessamento idempotente.
    """

    spark.sql(f"DROP TABLE IF EXISTS {GOLD_HOURLY_METRICS_TABLE}")
    spark.sql(f"DROP TABLE IF EXISTS {GOLD_DAILY_METRICS_TABLE}")

    print("Tabelas Gold antigas removidas, se existiam.")


def build_gold_daily_metrics() -> None:
    """
    Cria tabela Gold diária com métricas de consumo analítico.

    Granularidade:
    - pickup_date
    - pickup_month
    - taxi_type
    """

    df_silver = spark.table(SILVER_TABLE)

    df_gold_daily = (
        df_silver
        .groupBy(
            "pickup_date",
            "pickup_month",
            "taxi_type"
        )
        .agg(
            F.count("*").alias("total_trips"),
            F.sum("passenger_count").alias("total_passengers"),
            F.round(F.avg("passenger_count"), 2).alias("avg_passenger_count"),
            F.round(F.sum("total_amount"), 2).alias("sum_total_amount"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
            F.min("pickup_datetime").alias("first_pickup_datetime"),
            F.max("pickup_datetime").alias("last_pickup_datetime")
        )
        .withColumn("created_at", F.current_timestamp())
    )

    (
        df_gold_daily.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .partitionBy("pickup_month", "taxi_type")
        .saveAsTable(GOLD_DAILY_METRICS_TABLE)
    )

    print(f"Tabela Gold diária criada: {GOLD_DAILY_METRICS_TABLE}")


def build_gold_hourly_metrics() -> None:
    """
    Cria tabela Gold horária com métricas de consumo analítico.

    Granularidade:
    - pickup_date
    - pickup_month
    - pickup_hour
    - taxi_type
    """

    df_silver = spark.table(SILVER_TABLE)

    df_gold_hourly = (
        df_silver
        .groupBy(
            "pickup_date",
            "pickup_month",
            "pickup_hour",
            "taxi_type"
        )
        .agg(
            F.count("*").alias("total_trips"),
            F.sum("passenger_count").alias("total_passengers"),
            F.round(F.avg("passenger_count"), 2).alias("avg_passenger_count"),
            F.round(F.sum("total_amount"), 2).alias("sum_total_amount"),
            F.round(F.avg("total_amount"), 2).alias("avg_total_amount"),
            F.min("pickup_datetime").alias("first_pickup_datetime"),
            F.max("pickup_datetime").alias("last_pickup_datetime")
        )
        .withColumn("created_at", F.current_timestamp())
    )

    (
        df_gold_hourly.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .partitionBy("pickup_month", "taxi_type")
        .saveAsTable(GOLD_HOURLY_METRICS_TABLE)
    )

    print(f"Tabela Gold horária criada: {GOLD_HOURLY_METRICS_TABLE}")


def silver_to_gold() -> None:
    """
    Executa a geração da camada Gold.
    """

    print("Etapa Gold 1 - Validando tabela Silver")
    validate_silver_table_exists()

    print("Etapa Gold 2 - Limpando tabelas Gold anteriores")
    drop_gold_tables_if_exists()

    print("Etapa Gold 3 - Criando Gold diária")
    build_gold_daily_metrics()

    print("Etapa Gold 4 - Criando Gold horária")
    build_gold_hourly_metrics()

    print("Pipeline Gold executado com sucesso.")

In [0]:
silver_to_gold()